In [82]:
#Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as mlt
import seaborn as sns

In [83]:
#Reading dataset
deliveries=pd.read_csv("deliveries.csv")
matches=pd.read_csv("matches.csv")

In [84]:
matches.columns

Index(['id', 'season', 'city', 'date', 'team1', 'team2', 'toss_winner',
       'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs',
       'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2',
       'umpire3'],
      dtype='object')

In [85]:
matches = matches[
    (matches['result'] == 'normal') &
    (matches['dl_applied'] == 0)
]

In [86]:
matches=matches.drop(columns=[
    'umpire1',
    'umpire2',
    'umpire3',
    'player_of_match',
    'dl_applied',
    'result',
    'win_by_wickets',
    'win_by_runs',
    'city',
    'date'
])

In [87]:
matches.columns

Index(['id', 'season', 'team1', 'team2', 'toss_winner', 'toss_decision',
       'winner', 'venue'],
      dtype='object')

In [88]:
matches.head(89)

,id,season,team1,team2,toss_winner,toss_decision,winner,venue
0,1,2017,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Sunrisers Hyderabad,"Rajiv Gandhi International Stadium, Uppal"
1,2,2017,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,Rising Pune Supergiant,Maharashtra Cricket Association Stadium
2,3,2017,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,Kolkata Knight Riders,Saurashtra Cricket Association Stadium
3,4,2017,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,Kings XI Punjab,Holkar Cricket Stadium
4,5,2017,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,Royal Challengers Bangalore,M Chinnaswamy Stadium
...,...,...,...,...,...,...,...,...
86,87,2008,Delhi Daredevils,Chennai Super Kings,Chennai Super Kings,field,Chennai Super Kings,Feroz Shah Kotla
87,88,2008,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders,bat,Kolkata Knight Riders,Eden Gardens
88,89,2008,Deccan Chargers,Rajasthan Royals,Rajasthan Royals,field,Rajasthan Royals,Sawai Mansingh Stadium
89,90,2008,Royal Challengers Bangalore,Mumbai Indians,Mumbai Indians,field,Mumbai Indians,M Chinnaswamy Stadium


In [89]:
valid_match_ids = matches['id'].unique()

deliveries = deliveries[deliveries['match_id'].isin(valid_match_ids)]

In [90]:
deliveries['wicket'] = deliveries['player_dismissed'].notnull().astype(int)

In [91]:
deliveries['cumulative_wickets'] = deliveries.groupby(['match_id', 'inning'])['wicket'].cumsum()

In [92]:
deliveries['cumulative_runs'] = deliveries.groupby(['match_id', 'inning'])['total_runs'].cumsum()

In [93]:
deliveries=deliveries.drop(columns=[
    'fielder',
    'dismissal_kind',
    'is_super_over',
    'batsman',
    'non_striker',
    'bowler',
    'player_dismissed',
    'wide_runs',
    'bye_runs',
    'legbye_runs',
    'noball_runs',
    'penalty_runs'
])

In [94]:
deliveries.columns

Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batsman_runs', 'extra_runs', 'total_runs', 'wicket',
       'cumulative_wickets', 'cumulative_runs'],
      dtype='object')

In [95]:
deliveries.head(10)

,match_id,inning,batting_team,bowling_team,over,ball,batsman_runs,extra_runs,total_runs,wicket,cumulative_wickets,cumulative_runs
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,0,0,0,0,0,0
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,0,0,0,0,0,0
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,4,0,4,0,0,4
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,0,0,0,0,0,4
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,0,2,2,0,0,6
5,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,6,0,0,0,0,0,6
6,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,7,0,1,1,0,0,7
7,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,2,1,1,0,1,0,0,8
8,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,2,2,4,0,4,0,0,12
9,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,2,3,0,1,1,0,0,13


In [96]:
# Split into two innings
innings1 = deliveries[deliveries['inning'] == 1].copy()
innings2 = deliveries[deliveries['inning'] == 2].copy()

In [97]:
innings1['balls_bowled'] = ((innings1['over'] - 1) * 6) + innings1['ball']
innings1['current_run_rate'] = (innings1['cumulative_runs'] / innings1['balls_bowled']) * 6

In [98]:
# Get total score from innings 1 per match
innings1_totals = innings1.groupby('match_id')['total_runs'].sum().reset_index()
innings1_totals.columns = ['match_id', 'target']

# Merge target into innings 2
innings2 = innings2.merge(innings1_totals, on='match_id')

# Now derive chase features
innings2['balls_bowled'] = ((innings2['over'] - 1) * 6) + innings2['ball']
innings2['balls_remaining'] = 120 - innings2['balls_bowled']
innings2['runs_required'] = (innings2['target'] + 1) - innings2['cumulative_runs']
innings2['current_run_rate'] = (innings2['cumulative_runs'] / innings2['balls_bowled']) * 6
innings2['required_run_rate'] = (innings2['runs_required'] / innings2['balls_remaining']) * 6

In [99]:
innings1.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman_runs,extra_runs,total_runs,wicket,cumulative_wickets,cumulative_runs,balls_bowled,current_run_rate
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,0,0,0,0,0,0,1,0.0
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,0,0,0,0,0,0,2,0.0
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,4,0,4,0,0,4,3,8.0
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,0,0,0,0,0,4,4,6.0
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,0,2,2,0,0,6,5,7.2


In [100]:
innings2.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman_runs,extra_runs,total_runs,wicket,cumulative_wickets,cumulative_runs,target,balls_bowled,balls_remaining,runs_required,current_run_rate,required_run_rate
0,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,1,0,1,0,0,1,207,1,119,207,6.0,10.436975
1,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,0,0,0,0,0,1,207,2,118,207,3.0,10.525424
2,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,0,0,0,0,0,1,207,3,117,207,2.0,10.615385
3,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,2,0,2,0,0,3,207,4,116,205,4.5,10.603448
4,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,4,0,4,0,0,7,207,5,115,201,8.4,10.486957


In [101]:
# Get only id and winner from matches
match_winners = matches[['id', 'winner']].rename(columns={'id': 'match_id'})

# Merge
innings1 = innings1.merge(match_winners, on='match_id')
innings2 = innings2.merge(match_winners, on='match_id')

# Create target label
innings1['target_label'] = (innings1['batting_team'] == innings1['winner']).astype(int)
innings2['target_label'] = (innings2['batting_team'] == innings2['winner']).astype(int)

# Drop winner column
innings1.drop(columns=['winner'], inplace=True)
innings2.drop(columns=['winner'], inplace=True)

In [102]:
# Innings 1 drop
innings1.drop(columns=['match_id', 'inning', 'ball', 'batsman_runs', 'total_runs', 'wicket'], inplace=True)

# Innings 2 drop
innings2.drop(columns=['match_id', 'inning', 'ball', 'batsman_runs', 'total_runs', 'wicket'], inplace=True)

In [103]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Fit on all teams combined so both columns get same encoding
all_teams = pd.concat([innings1['batting_team'], innings1['bowling_team']]).unique()
le.fit(all_teams)

# Apply to innings1
innings1['batting_team'] = le.transform(innings1['batting_team'])
innings1['bowling_team'] = le.transform(innings1['bowling_team'])

# Apply to innings2
innings2['batting_team'] = le.transform(innings2['batting_team'])
innings2['bowling_team'] = le.transform(innings2['bowling_team'])

In [104]:
innings1.dropna(inplace=True)
innings2.dropna(inplace=True)

In [105]:
innings2.replace([np.inf, -np.inf], np.nan, inplace=True)
innings2.dropna(inplace=True)

In [106]:
innings1.head()

,batting_team,bowling_team,over,extra_runs,cumulative_wickets,cumulative_runs,balls_bowled,current_run_rate,target_label
0,13,12,1,0,0,0,1,0.0,1
1,13,12,1,0,0,0,2,0.0,1
2,13,12,1,0,0,4,3,8.0,1
3,13,12,1,0,0,4,4,6.0,1
4,13,12,1,2,0,6,5,7.2,1


In [107]:
innings2.head()

,batting_team,bowling_team,over,extra_runs,cumulative_wickets,cumulative_runs,target,balls_bowled,balls_remaining,runs_required,current_run_rate,required_run_rate,target_label
0,12,13,1,0,0,1,207,1,119,207,6.0,10.436975,0
1,12,13,1,0,0,1,207,2,118,207,3.0,10.525424,0
2,12,13,1,0,0,1,207,3,117,207,2.0,10.615385,0
3,12,13,1,0,0,3,207,4,116,205,4.5,10.603448,0
4,12,13,1,0,0,7,207,5,115,201,8.4,10.486957,0


In [108]:
# Train test split
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'XGBoost': XGBClassifier()
}
X1 = innings1.drop(columns=['target_label'])
y1 = innings1['target_label']
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

X2 = innings2.drop(columns=['target_label'])
y2 = innings2['target_label']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

# Model training
print("INNINGS 1")
for name, model in models.items():
    model.fit(X1_train, y1_train)
    y_pred = model.predict(X1_test)
    print(f"{name}: {accuracy_score(y1_test, y_pred)*100:.2f}%")

print("\nINNINGS 2")
for name, model in models.items():
    model.fit(X2_train, y2_train)
    y_pred = model.predict(X2_test)
    print(f"{name}: {accuracy_score(y2_test, y_pred)*100:.2f}%")

INNINGS 1
Logistic Regression: 62.25%
Decision Tree: 89.23%
Random Forest: 90.09%
XGBoost: 83.97%

INNINGS 2


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression: 79.01%
Decision Tree: 97.62%
Random Forest: 99.09%
XGBoost: 99.79%


In [109]:
rf_model1 = RandomForestClassifier()
rf_model1.fit(X1_train, y1_train)

xgb_model2 = XGBClassifier()
xgb_model2.fit(X2_train, y2_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [110]:
def predict_win_probability(innings, batting_team, bowling_team, over, balls_bowled,
                             cumulative_runs, cumulative_wickets, extra_runs,
                             current_run_rate, target=None, balls_remaining=None,
                             runs_required=None, required_run_rate=None):

    # Encode teams
    batting_encoded = le.transform([batting_team])[0]
    bowling_encoded = le.transform([bowling_team])[0]

    if innings == 1:
        input_data = pd.DataFrame({
            'batting_team': [batting_encoded],
            'bowling_team': [bowling_encoded],
            'over': [over],
            'extra_runs': [extra_runs],
            'cumulative_wickets': [cumulative_wickets],
            'cumulative_runs': [cumulative_runs],
            'balls_bowled': [balls_bowled],
            'current_run_rate': [current_run_rate]
        })
        prob = rf_model1.predict_proba(input_data)[0]

    else:
        input_data = pd.DataFrame({
            'batting_team': [batting_encoded],
            'bowling_team': [bowling_encoded],
            'over': [over],
            'extra_runs': [extra_runs],
            'cumulative_wickets': [cumulative_wickets],
            'cumulative_runs': [cumulative_runs],
            'target': [target],
            'balls_bowled': [balls_bowled],
            'balls_remaining': [balls_remaining],
            'runs_required': [runs_required],
            'current_run_rate': [current_run_rate],
            'required_run_rate': [required_run_rate]
        })
        prob = xgb_model2.predict_proba(input_data)[0]

    print(f"Batting Team ({batting_team}) Win Probability: {prob[1]*100:.2f}%")
    print(f"Bowling Team ({bowling_team}) Win Probability: {prob[0]*100:.2f}%")

In [112]:
# 200/0 after 18 overs, no wickets lost
predict_win_probability(
    innings=1, batting_team='Mumbai Indians', bowling_team='Chennai Super Kings',
    over=18, balls_bowled=108, cumulative_runs=200, cumulative_wickets=0,
    extra_runs=0, current_run_rate=11.1
)

# Chasing 250, need 150 off 12 balls with 9 wickets gone
predict_win_probability(
    innings=2, batting_team='Mumbai Indians', bowling_team='Chennai Super Kings',
    over=19, balls_bowled=108, cumulative_runs=101, cumulative_wickets=9,
    extra_runs=0, current_run_rate=5.6, target=250,
    balls_remaining=12, runs_required=150, required_run_rate=75.0
)


predict_win_probability(
    innings=2, batting_team='Mumbai Indians', bowling_team='Chennai Super Kings',
    over=17, balls_bowled=102, cumulative_runs=119, cumulative_wickets=4,
    extra_runs=5, current_run_rate=7.0, target=160,
    balls_remaining=18, runs_required=48, required_run_rate=16.0
)


predict_win_probability(
    innings=2, batting_team='Mumbai Indians', bowling_team='Chennai Super Kings',
    over=18, balls_bowled=112, cumulative_runs=138, cumulative_wickets=4,
    extra_runs=5, current_run_rate=7.39, target=160,
    balls_remaining=8, runs_required=28, required_run_rate=21.0
)

Batting Team (Mumbai Indians) Win Probability: 95.00%
Bowling Team (Chennai Super Kings) Win Probability: 5.00%
Batting Team (Mumbai Indians) Win Probability: 0.00%
Bowling Team (Chennai Super Kings) Win Probability: 100.00%
Batting Team (Mumbai Indians) Win Probability: 11.91%
Bowling Team (Chennai Super Kings) Win Probability: 88.09%
Batting Team (Mumbai Indians) Win Probability: 9.37%
Bowling Team (Chennai Super Kings) Win Probability: 90.63%
